In [1]:
# Association Rules - Market Basket Analysis
# Dataset: Groceries dataset - https://www.kaggle.com/datasets/heeraldedhia/groceries-dataset
# (simpler alternative to Instacart, which is much larger and needs more setup)
#
# Needs: pip install mlxtend

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_csv("../Datasets/Groceries_dataset.csv")


In [2]:
# ---- 1. First look ----
print(df.shape)
print(df.head())

# columns are usually: Member_number, Date, itemDescription
# each row = one item bought by one member on one date

print(df.isnull().sum())

# ---- 2. Group items into transactions ----
# a "transaction" here = everything one member bought on one date
df["Transaction"] = df["Member_number"].astype(str) + "_" + df["Date"].astype(str)

transactions = df.groupby("Transaction")["itemDescription"].apply(list)
print(transactions.head())
print("Number of transactions:", len(transactions))

(38765, 3)
   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk
Member_number      0
Date               0
itemDescription    0
dtype: int64
Transaction
1000_15-03-2015    [sausage, whole milk, semi-finished bread, yog...
1000_24-06-2014                    [whole milk, pastry, salty snack]
1000_24-07-2015                       [canned beer, misc. beverages]
1000_25-11-2015                          [sausage, hygiene articles]
1000_27-05-2015                           [soda, pickled vegetables]
Name: itemDescription, dtype: object
Number of transactions: 14963


In [3]:
# ---- 3. Convert to one-hot encoded format ----
# apriori needs a table where each column = an item, each row = a transaction,
# with True/False showing if that item was in that transaction
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print(basket_df.shape)
print(basket_df.head())

# ---- 4. Find frequent itemsets ----
# min_support = how often an itemset needs to appear to be considered "frequent"
# start low since grocery data is usually sparse (many unique items, few repeats)
frequent_itemsets = apriori(basket_df, min_support=0.001, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False)

print(frequent_itemsets.head(15))

(14963, 167)
   Instant food products  UHT-milk  abrasive cleaner  artif. sweetener  \
0                  False     False             False             False   
1                  False     False             False             False   
2                  False     False             False             False   
3                  False     False             False             False   
4                  False     False             False             False   

   baby cosmetics   bags  baking powder  bathroom cleaner   beef  berries  \
0           False  False          False             False  False    False   
1           False  False          False             False  False    False   
2           False  False          False             False  False    False   
3           False  False          False             False  False    False   
4           False  False          False             False  False    False   

   ...  turkey  vinegar  waffles  whipped/sour cream  whisky  white bread  \
0 

In [4]:
# ---- 5. Generate association rules ----
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False)

print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15))

# lift > 1 means the items are bought together MORE than random chance would suggest
# confidence = "if they bought A, how often did they also buy B"

# ---- 6. Look at a specific example ----
# find rules where "whole milk" is the antecedent (since it's usually the most common item)
milk_rules = rules[rules["antecedents"].apply(lambda x: "whole milk" in x)]
print(milk_rules[["antecedents", "consequents", "confidence", "lift"]].head(10))


                          antecedents                       consequents  \
80    frozenset({whole milk, yogurt})              frozenset({sausage})   
85               frozenset({sausage})   frozenset({whole milk, yogurt})   
81   frozenset({whole milk, sausage})               frozenset({yogurt})   
84                frozenset({yogurt})  frozenset({whole milk, sausage})   
114  frozenset({specialty chocolate})         frozenset({citrus fruit})   
115         frozenset({citrus fruit})  frozenset({specialty chocolate})   
82       frozenset({yogurt, sausage})           frozenset({whole milk})   
83            frozenset({whole milk})      frozenset({yogurt, sausage})   
191       frozenset({tropical fruit})                frozenset({flour})   
190                frozenset({flour})       frozenset({tropical fruit})   
76             frozenset({beverages})              frozenset({sausage})   
77               frozenset({sausage})            frozenset({beverages})   
213              frozense

In [5]:
# ---- 7. Save rules ----
rules.to_csv("association_rules_output.csv", index=False)
print("Saved -> association_rules_output.csv")

# Note: Instacart Market Basket Analysis follows the exact same apriori/association_rules
# approach - the only real difference is it's a much bigger dataset (millions of orders
# across multiple files that need merging first), so it's worth trying this Groceries
# version first to get the concepts down before scaling up to Instacart.

Saved -> association_rules_output.csv
